# 🫁 TB IGRA Longitudinal Study — Data Analysis
### Cohort Exploration, Feature Engineering & Visualization

**Author:** Saloni Prasad  
**Dataset:** `case_study.csv` — 149 subjects × 23 columns — 24-month longitudinal TB cohort  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn  

---

### Notebook Structure
1. Data Loading & Initial Exploration
2. Cohort Distribution Overview
3. Longitudinal IGRA Trajectory Analysis
4. Key Risk Factor Insights
5. Feature Engineering (BMI Categories + IFN-Gamma Delta)
6. Summary Statistics on Engineered Features
7. Visualizations

---
## 1. Data Loading & Initial Exploration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('../data/raw/case_study.csv')

# Strip whitespace from column names
df.columns = df.columns.str.strip()

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

Dataset shape: (149, 23)
Columns: ['Baseline', 'AGE', 'GENDER', 'HT', 'WT', 'BMI', 'Baseline IGRA', 'Month-6', 'Month-6 IGRA', 'Month-12', 'Month-12 IGRA', 'Month- 18', 'Month- 18 IGRA', 'Month-24', 'Month24 IGRA', 'BCG VACCINATION', 'OUTCOME', 'IFN-GAMMA-UNS', 'IFN-GAMMA-C+E', 'SMOKING', 'STATUS', 'DIABETES STATUS', 'REGION']


,Baseline,AGE,GENDER,HT,WT,BMI,Baseline IGRA,Month-6,Month-6 IGRA,OUTCOME,SMOKING,STATUS,DIABETES STATUS,REGION
0,TB-01,44,MALE,175,84.0,27.43,POSITIVE,TB-150,INDETERMINATE,NON-PROGRESSOR,Yes,Latent,Yes,Rural
1,TB-02,36,Female,165,60.0,22.04,NEGATIVE,TB-151,NEGATIVE,NON-CONVERTER,No,Latent,Yes,Rural
2,TB-03,14,MALE,137,58.0,30.90,POSITIVE,TB-152,POSITIVE,NON-PROGRESSOR,Yes,Latent,Yes,Rural
3,TB-04,12,MALE,132,55.0,31.57,NEGATIVE,TB-153,POSITIVE,PROGRESSOR,Yes,Active,Yes,Rural
4,TB-05,50,Female,161,55.0,21.22,POSITIVE,TB-154,POSITIVE,REVERTER,No,Latent,Yes,Rural


---
## 2. Cohort Distribution Overview

In [2]:
# Print distribution of key categorical variables
print('=== COHORT DISTRIBUTION ===')
print(f"\nOutcome Groups:      {df['OUTCOME'].value_counts().to_dict()}")
print(f"TB Status:           {df['STATUS'].value_counts().to_dict()}")
print(f"Smoking:             {df['SMOKING'].value_counts().to_dict()}")
print(f"Diabetes:            {df['DIABETES STATUS'].value_counts().to_dict()}")
print(f"BCG Vaccination:     {df['BCG VACCINATION'].value_counts().to_dict()}")
print(f"Region:              {df['REGION'].value_counts().to_dict()}")

=== COHORT DISTRIBUTION ===

Outcome Groups:      {'NON-CONVERTER': 67, 'NON-PROGRESSOR': 41, 'PROGRESSOR': 15, 'REVERTER': 15, 'CONVERTER': 11}
TB Status:           {'Latent': 135, 'Active': 14}
Smoking:             {'Yes': 100, 'No': 49}
Diabetes:            {'Yes': 113, 'No': 36}
BCG Vaccination:     {'Yes': 128, 'No': 21}
Region:              {'Urban': 77, 'Rural': 72}


---
## 3. Longitudinal IGRA Trajectory Analysis

In [3]:
# Mode IGRA status across all 5 timepoints per outcome group
igra_cols = ['Baseline IGRA', 'Month-6 IGRA', 'Month-12 IGRA', 'Month- 18 IGRA', 'Month24 IGRA']

print('=== IGRA STATUS MODE BY OUTCOME × TIMEPOINT ===')
print(df.groupby('OUTCOME')[igra_cols].agg(lambda x: x.mode()[0] if not x.empty else ''))

# Cross-tabulation: Outcome vs TB Status
print('\n=== TB STATUS vs OUTCOME ===')
print(pd.crosstab(df['OUTCOME'], df['STATUS']))

# Mean biomarker and clinical metrics per outcome group
print('\n=== MEAN IFN-GAMMA, BMI & AGE BY OUTCOME ===')
print(df.groupby('OUTCOME')[['IFN-GAMMA-UNS', 'IFN-GAMMA-C+E', 'BMI', 'AGE']].mean().round(2))

=== IGRA STATUS MODE BY OUTCOME × TIMEPOINT ===
               Baseline IGRA Month-6 IGRA Month-12 IGRA Month- 18 IGRA Month24 IGRA
OUTCOME                                                                            
CONVERTER           NEGATIVE     POSITIVE      POSITIVE       POSITIVE     POSITIVE
NON-CONVERTER       NEGATIVE     NEGATIVE      NEGATIVE       NEGATIVE     NEGATIVE
NON-PROGRESSOR      POSITIVE     POSITIVE      POSITIVE       POSITIVE     POSITIVE
PROGRESSOR          POSITIVE     POSITIVE      POSITIVE       POSITIVE     POSITIVE
REVERTER            POSITIVE     NEGATIVE      NEGATIVE       NEGATIVE     NEGATIVE

=== TB STATUS vs OUTCOME ===
STATUS          Active  Latent
OUTCOME                       
CONVERTER            0      11
NON-CONVERTER        0      67
NON-PROGRESSOR       0      41
PROGRESSOR          14       1
REVERTER             0      15

=== MEAN IFN-GAMMA, BMI & AGE BY OUTCOME ===
                IFN-GAMMA-UNS  IFN-GAMMA-C+E    BMI    AGE
OUTCOME     

---
## 4. Key Risk Factor Insights

In [4]:
# Compute percentage statistics for key findings
total = len(df)
progressors = df[df['OUTCOME'] == 'PROGRESSOR']

prog_diabetes = len(progressors[progressors['DIABETES STATUS'] == 'Yes'])
prog_smoking  = len(progressors[progressors['SMOKING'] == 'Yes'])
prog_bcg      = len(progressors[progressors['BCG VACCINATION'] == 'Yes'])
active_tb     = len(df[df['STATUS'] == 'Active'])

print('=' * 55)
print('          KEY RISK FACTOR INSIGHTS')
print('=' * 55)
print(f'Total Cohort Size          : {total}')
print(f'Active TB Cases            : {active_tb} ({active_tb/total*100:.1f}%)')
print(f'Progressor Group Size      : {len(progressors)}')
print()
print(f'Progressors with Diabetes  : {prog_diabetes}/{len(progressors)} ({prog_diabetes/len(progressors)*100:.1f}%)')
print(f'Progressors with Smoking   : {prog_smoking}/{len(progressors)} ({prog_smoking/len(progressors)*100:.1f}%)')
print(f'Progressors with BCG       : {prog_bcg}/{len(progressors)} ({prog_bcg/len(progressors)*100:.1f}%)')
print('=' * 55)
print()
print('KEY FINDING: 100% of PROGRESSORs had BOTH Diabetes AND Smoking.')
print('             14/15 Progressors developed Active TB (93.3%).')

          KEY RISK FACTOR INSIGHTS
Total Cohort Size          : 149
Active TB Cases            : 14 (9.4%)
Progressor Group Size      : 15

Progressors with Diabetes  : 15/15 (100.0%)
Progressors with Smoking   : 15/15 (100.0%)
Progressors with BCG       : 15/15 (100.0%)

KEY FINDING: 100% of PROGRESSORs had BOTH Diabetes AND Smoking.
             14/15 Progressors developed Active TB (93.3%).


---
## 5. Feature Engineering

In [5]:
# Standardize gender casing
df['GENDER'] = df['GENDER'].str.upper()

# BMI Category Classification (WHO Standard)
def classify_bmi(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif 18.5 <= bmi < 25.0:
        return 'Normal'
    elif 25.0 <= bmi < 30.0:
        return 'Overweight'
    else:
        return 'Obese'

df['BMI_CATEGORY'] = df['BMI'].apply(classify_bmi)

# Biomarker Response Delta (Stimulated - Unstimulated IFN-Gamma)
df['IFN_GAMMA_DELTA'] = df['IFN-GAMMA-C+E'] - df['IFN-GAMMA-UNS']

# Export processed dataset
df.to_csv('../data/processed/processed_tb_cohort_analysis.csv', index=False)
print('Feature engineering complete.')
print('Processed dataset saved to: ../data/processed/processed_tb_cohort_analysis.csv')
print()
print('New features added:')
print('  BMI_CATEGORY     — WHO standard classification (Underweight/Normal/Overweight/Obese)')
print('  IFN_GAMMA_DELTA  — Stimulated minus Unstimulated IFN-Gamma (pg/mL)')

Feature engineering complete.
Processed dataset saved to: ../data/processed/processed_tb_cohort_analysis.csv

New features added:
  BMI_CATEGORY     — WHO standard classification (Underweight/Normal/Overweight/Obese)
  IFN_GAMMA_DELTA  — Stimulated minus Unstimulated IFN-Gamma (pg/mL)


---
## 6. Summary Statistics on Engineered Features

In [6]:
# Load the processed dataset
df_processed = pd.read_csv('../data/processed/processed_tb_cohort_analysis.csv')

# BMI Category distribution
print('=== BMI CATEGORY DISTRIBUTION ===')
print(df_processed['BMI_CATEGORY'].value_counts())
print()

# IFN-Gamma Delta statistics
print('=== IFN_GAMMA_DELTA STATISTICS ===')
print(df_processed['IFN_GAMMA_DELTA'].describe().round(3))

=== BMI CATEGORY DISTRIBUTION ===
Normal         64
Overweight     38
Underweight    37
Obese          10
Name: BMI_CATEGORY, dtype: int64

=== IFN_GAMMA_DELTA STATISTICS ===
count     149.000
mean      128.070
std       306.840
min      -492.660
25%         0.098
50%        17.904
75%        83.268
max      1297.420
Name: IFN_GAMMA_DELTA, dtype: float64


---
## 7. Visualizations

In [7]:
# Visual 1: Distribution of BMI Categories
plt.figure(figsize=(8, 6))
sns.countplot(
    data=df_processed,
    x='BMI_CATEGORY',
    order=df_processed['BMI_CATEGORY'].value_counts().index,
    hue='BMI_CATEGORY',
    palette='viridis',
    legend=False
)
plt.title('Distribution of BMI Categories', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('BMI Category', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [8]:
# Visual 2: Age vs IFN-Gamma Delta by BMI Category
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=df_processed,
    x='AGE',
    y='IFN_GAMMA_DELTA',
    hue='BMI_CATEGORY',
    palette='viridis',
    s=100,
    alpha=0.7
)
plt.axhline(0, color='red', linestyle='--', linewidth=1, label='No net response (delta = 0)')
plt.title('AGE vs. IFN-Gamma Delta\n(Stimulated − Unstimulated) by BMI Category',
          fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Age (years)', fontsize=12)
plt.ylabel('IFN-Gamma Delta (pg/mL)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(title='BMI Category', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()